In [ ]:
# Lab type: review
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Classical Models — Smoothing and ARIMA, When They Win
# Task: The two model fits below are correctly implemented and reproduce the
#       lesson's results. Answer the judgment questions — each is a decision a
#       practitioner (or an AI assistant) has to defend. Write 2-4 sentences each.

# Lab: Judging Two Fitted Forecasters

Holt-Winters and SARIMA, fit exactly as in the lesson. The code is correct; the
judgment calls around it are yours to examine.

**Outputs are cleared.** Run each cell to generate results (the SARIMA fit takes
~30 seconds).

## Setup and fits

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

train, test = orders.iloc[:-90], orders.iloc[-90:]

hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()
hw_fc = hw.forecast(90)

sarima = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
sarima_fc = sarima.forecast(90)

def mape(fc): return ((test - fc).abs() / test).mean() * 100
print(f"Holt-Winters MAPE: {mape(hw_fc):.1f}%")
print(f"SARIMA MAPE:       {mape(sarima_fc):.1f}%")
print(f"\nHW smoothing params: alpha={hw.params['smoothing_level']:.3f}, "
      f"beta={hw.params['smoothing_trend']:.4f}, gamma={hw.params['smoothing_seasonal']:.3f}")

**Question 1.** The fitted Holt-Winters has `alpha ≈ 0.08` and `gamma ≈ 0.00` — both
very low. In plain language, what is the model claiming about this series? Describe a
business scenario (a change in the store's situation) where these fitted values would
become exactly the wrong configuration, and what you'd expect to see in the forecasts
when it happened.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 1</summary>

Low alpha: the level barely reacts to individual days — the model treats daily
fluctuations as noise around a slowly-moving level. Gamma ≈ 0: the weekly shape is
treated as fixed for all three years. That's correct for this stable synthetic series.
It becomes wrong the moment the process shifts — e.g. a rebrand or a new fulfilment
partner changes the weekend pattern, or a viral moment steps the level up. A low-alpha,
zero-gamma model would keep forecasting the *old* level and *old* weekly shape for
weeks, with persistent one-sided errors (residuals all positive or all negative) as the
tell-tale symptom.

</details>

**Question 2.** Run the residual diagnostics below, then judge: is there anything left
for a more complex model to find? What specific number would change your mind?

In [ ]:
resid = pd.Series(hw.resid, index=train.index)
print(f"residual std:            {resid.std():.1f}")
print(f"residual lag-1 autocorr: {resid.autocorr(1):.3f}")
print(f"residual lag-7 autocorr: {resid.autocorr(7):.3f}")
print(f"(the dataset was built with noise sigma = 16)")

<details>
<summary>🔑 Reveal answer — Question 2</summary>

The residual std is close to the known noise floor (σ=16) and both autocorrelations are
near zero — the model has extracted essentially all the forecastable structure, so a
more complex model has nothing legitimate left to find. What would change the verdict:
meaningful residual autocorrelation (say |r| > 0.1–0.15 at lag 1, 7, or ~365) or
residual spread well above the noise floor. On real data you don't know σ, so the
autocorrelation check carries the weight — plus the backtest of Lesson 7.

</details>

**Question 3.** An AI assistant, asked to improve on the SARIMA result, proposes
`SARIMA(1,1,1)(1,1,1)₃₆₅` — "to capture the yearly seasonality as well." The idea is
directionally reasonable; the specific proposal is a bad one. Give two independent
reasons it will fail in practice, and name a better route to capturing the yearly
component with the tools from Lessons 3–5.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 3</summary>

(1) Computational/statistical: seasonal differencing and seasonal AR/MA at lag 365
means the model effectively estimates from pairs of observations a full year apart —
with 3 years of data that's ~2 usable cycles, far too few to estimate the seasonal
parameters, and the state-space fit becomes enormous and slow. (2) The seasonal
difference at 365 throws away a year of data outright and interacts badly with the
weekly structure the m=7 terms were handling. Better routes: model the deseasonalised
series (remove the yearly component via decomposition at weekly grain / STL, forecast
the remainder, add the component back), or use exogenous Fourier terms for the yearly
cycle in SARIMAX — both express "smooth yearly wave" with a handful of parameters.

</details>

**Question 4.** Your stakeholder needs a *14-day* forecast, updated weekly. Based on
everything in this lab and the lesson's discussion of horizons, which of the two fitted
models would you put forward, and what evaluation (be specific about design) would you
run before committing?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 4</summary>

Don't decide from the single 90-day comparison — the 14-day task is a different
horizon, and ARIMA's flattening weakness shrinks at short horizons. The evaluation:
a rolling-origin backtest (Lesson 7) with 14-day test windows stepped weekly across
at least a year, reporting per-fold MAPE for Holt-Winters, SARIMA, *and the seasonal
naive*, then choose on mean and worst-fold error. The defensible prior is Holt-Winters
(it won the long horizon and its residuals are clean), but the honest answer is a
design, not a pick.

</details>

## Summary

> **Complete each sentence in one line.**

1. Low fitted alpha/gamma means the model claims the level and seasonal shape are ________.
2. A clean residual ACF near the noise floor means a more complex model ________.
3. Model choice is horizon-specific, so the deciding evidence is ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. ...are **stable — slow-moving level, essentially fixed weekly pattern** (and the
   model will lag behind any regime change).
2. A more complex model **has no legitimate signal left to win — it can only overfit
   noise or leak**.
3. The deciding evidence is **a rolling-origin backtest at the deployment horizon, with
   the seasonal naive alongside**.

</details>